<a href="https://colab.research.google.com/github/abhimanyu1502/flyrank1st-assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. My Lane as an ML Task (Type)

**Task Formulation**: **Unsupervised Clustering & Archetype Profiling** (K-Means / GMM with PCA)

- **Why Clustering**: Enterprise content repositories do not possess pre-labeled "archetype" ground truth. The goal is to discover natural, multivariate groupings in observable search telemetry.
- **Why Not Pure Ranking**: A 1D priority ranking tells you *which* page to touch first, but not *what* action to take. Clustering groups pages into qualitative behavioral personas (e.g., *Stale High-Reach* vs. *Hidden Gem* vs. *Low Engagement*), allowing each cluster to trigger a tailored editorial playbook.

In [3]:
import os
import pandas as pd
import numpy as np

# 1. Dynamic path resolution across environments
possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/content_refresh_anonymized.csv",
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
]
data_path = next((p for p in possible_paths if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df_raw = pd.read_csv(data_path)
df_clean = df_raw[(df_raw['impressions_90d'] >= 10) & (df_raw['content_age_days'] >= 90)].copy()

print("=== ML TASK DEFINITION ===")
print(f"Task Type:        Unsupervised Clustering & Archetype Segmentation")
print(f"Input Corpus:     {len(df_clean):,} active content pages across {df_clean['client_id'].nunique()} client sites")
print(f"Algorithm Family: Distance-based Partitioning (K-Means, PCA, GMM)")

=== ML TASK DEFINITION ===
Task Type:        Unsupervised Clustering & Archetype Segmentation
Input Corpus:     26,254 active content pages across 31 client sites
Algorithm Family: Distance-based Partitioning (K-Means, PCA, GMM)


### 2. Target or Proxy

- **Model Output**: A discrete cluster assignment `archetype_cluster` $\in \{0, 1, 2, 3, 4\}$ mapping to business archetypes:
  - *Cluster 0: High-Performing Champions*
  - *Cluster 1: Stale High-Reach Candidates*
  - *Cluster 2: Hidden Gems (High CTR, Mid Position)*
  - *Cluster 3: Low-Engagement / High-Bounce*
  - *Cluster 4: Thin / Low-Demand Content*
- **Validation Proxy**: Since true archetype labels are unobserved, we evaluate cluster quality using **Downstream Traffic Decline Rate** (`trend_direction == 'down'`). A valid clustering model must produce clusters with statistically distinct decline rates without ever training on the decline label.

In [7]:
# Sketching the Target / Proxy Column
# Construct an observable validation proxy: binary traffic decline flag
df_clean['decline_proxy_target'] = (df_clean['trend_direction'] == 'down').astype(int)

# Sketching the archetype target taxonomy
archetype_mapping = {
    0: "Champion (High Reach, Top Rank, Fresh)",
    1: "Stale High-Reach (High Reach, Aging)",
    2: "Hidden Gem (Mid Rank, Strong CTR)",
    3: "Low Engagement (Visible, Weak Retention)",
    4: "Low Demand (Low Volume, Thin Reach)"
}

print("=== TARGET / PROXY SKETCH ===")
print("Learned Model Target: 'archetype_cluster' (Categorical Integer 0-4)")
print(f"Validation Proxy:     'decline_proxy_target' (Binary 0/1, Base Rate = {df_clean['decline_proxy_target'].mean():.1%})")
print("\nArchetype Taxonomy:")
for k, v in archetype_mapping.items():
    print(f"  Cluster {k}: {v}")

=== TARGET / PROXY SKETCH ===
Learned Model Target: 'archetype_cluster' (Categorical Integer 0-4)
Validation Proxy:     'decline_proxy_target' (Binary 0/1, Base Rate = 59.1%)

Archetype Taxonomy:
  Cluster 0: Champion (High Reach, Top Rank, Fresh)
  Cluster 1: Stale High-Reach (High Reach, Aging)
  Cluster 2: Hidden Gem (Mid Rank, Strong CTR)
  Cluster 3: Low Engagement (Visible, Weak Retention)
  Cluster 4: Low Demand (Low Volume, Thin Reach)


### 3. Success Metric

We defend two complementary success metrics:

1. **Geometric Clustering Metric**: **Silhouette Score $\ge 0.35$** (and Davies-Bouldin Index $< 1.20$) computed on normalized feature embeddings. This confirms that clusters represent mathematically distinct, cohesive groupings rather than arbitrary partitions.
2. **Business Separation Metric**: **Cluster Decline Rate Spread $\ge 15.0\%$**. The difference in percentage of declining pages between the most vulnerable cluster and the healthiest cluster must exceed 15 percentage points, proving that the archetypes carry distinct risk profiles.

In [13]:
# Metric Defense & Benchmark Thresholds
success_criteria = pd.DataFrame({
    'Metric Type': ['Geometric Quality', 'Geometric Separation', 'Business Utility'],
    'Metric Name': ['Silhouette Score', 'Davies-Bouldin Index', 'Cluster Decline Rate Spread'],
    'Defensible Target': ['>= 0.35', '< 1.20', '>= 15.0% Spread'],
    'Failure Condition': ['< 0.20 (Overlapping clusters)', '> 1.80 (Poor separation)', '< 5.0% Spread (No behavioral difference)']
})

print("=== DEFENDED SUCCESS METRICS ===")
print(success_criteria.to_string(index=False))

=== DEFENDED SUCCESS METRICS ===
         Metric Type                 Metric Name Defensible Target                        Failure Condition
   Geometric Quality            Silhouette Score           >= 0.35            < 0.20 (Overlapping clusters)
Geometric Separation        Davies-Bouldin Index            < 1.20                 > 1.80 (Poor separation)
    Business Utility Cluster Decline Rate Spread   >= 15.0% Spread < 5.0% Spread (No behavioral difference)


# 4. The Unit of Analysis, as a Real DataFrame

- **Unit of Analysis**: **One row = One unique published content URL / page (`content_id`)**.
- **Granularity**: All search impressions, clicks, ranking positions, staleness metrics, and user engagement rates are aggregated to this exact URL level over a rolling 90-day observation window.

In [14]:
# Displaying the exact unit of analysis dataframe slice
unit_cols = [
    'content_id', 'client_id', 'content_type',
    'impressions_90d', 'clicks_90d', 'avg_position',
    'ctr', 'days_since_last_update', 'engagement_rate', 'content_age_days'
]

df_unit = df_clean[unit_cols].head(5).copy()

print("=== UNIT OF ANALYSIS: 1 ROW = 1 PUBLISHED CONTENT ITEM ===")
print(f"Total Portfolio Rows: {len(df_clean):,}")
print(f"Unique Content IDs:   {df_clean['content_id'].nunique():,} (Verified 1:1 match)\n")
print(df_unit.to_string(index=False))

=== UNIT OF ANALYSIS: 1 ROW = 1 PUBLISHED CONTENT ITEM ===
Total Portfolio Rows: 26,254
Unique Content IDs:   26,254 (Verified 1:1 match)

          content_id         client_id    content_type  impressions_90d  clicks_90d  avg_position  ctr  days_since_last_update  engagement_rate  content_age_days
content_304f48230142 client_f369cb89fc keyword article             3803          29          10.6 0.76                      20             5.88               187
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7          20.3 0.05                      25             0.00               445
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          11          36.5 0.09                      20             0.00               141
content_331d6c4de07b client_19581e27de keyword article            11751          58           6.2 0.49                      22             1.28               463
content_d99b7a2d90ca client_3fdba35f04 keyword arti

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### 5. Why ML Beats a Fixed Rule Here

Fixed heuristic rules (nested `if/else` conditions) fail for content portfolio segmentation for three core reasons:

1. **Combinatorial Rule Explosion**: Evaluating 5 continuous performance dimensions (`impressions`, `position`, `CTR`, `staleness`, `engagement`) across just 3 threshold tiers (low/med/high) produces $3^5 = 243$ edge-case branches. Human-coded rules become impossible to maintain or tune.
2. **Rigid Threshold Fragility**: A hard rule like `impressions >= 500` creates artificial cliffs where a page with 499 impressions is treated radically differently from one with 501. Clustering computes continuous Euclidean proximity to archetype centroids.
3. **Multivariate Trade-offs**: Machine learning balances complex trade-offs simultaneously (e.g., compensating lower ranking position with exceptional click-through efficiency), capturing nuanced "Hidden Gem" archetypes that simplistic thresholds miss.

In [15]:
# Demonstrating the limitation of simplistic threshold rules vs. multi-dimensional variation
# Rule-based attempt: simple high-reach stale rule
rule_stale_champion = df_clean[
    (df_clean['impressions_90d'] >= 1000) &
    (df_clean['avg_position'] <= 10) &
    (df_clean['days_since_last_update'] >= 180)
]

print("=== LIMITATIONS OF HARDCODED RULES ===")
print(f"Pages matching rigid 'Stale Champion' rule (Imp>=1000, Pos<=10, Age>=180d): {len(rule_stale_champion)}")
print("A fixed threshold captures only a tiny fraction of edge cases.")
print("Unsupervised clustering evaluates multivariate Euclidean proximity, assigning 100% of pages to their nearest archetype centroid.")
print("\n[PASSED] Task framing and ML justification complete.")

=== LIMITATIONS OF HARDCODED RULES ===
Pages matching rigid 'Stale Champion' rule (Imp>=1000, Pos<=10, Age>=180d): 1
A fixed threshold captures only a tiny fraction of edge cases.
Unsupervised clustering evaluates multivariate Euclidean proximity, assigning 100% of pages to their nearest archetype centroid.

[PASSED] Task framing and ML justification complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.